In [ ]:
import pandas as pd
import numpy as np
from scipy.optimize import milp, LinearConstraint, Bounds

# ==========================================
# Session 1: Load & Prepare Data
# ==========================================
def clean_df(df):
    df.columns = df.columns.str.strip()
    for col in df.select_dtypes(include=['object']).columns:
        df[col] = df[col].astype(str).str.strip()
    return df

inv_df = clean_df(pd.read_csv('01-inventory.csv'))
recipe_df = clean_df(pd.read_csv('02-recipe.csv'))
recipe_sum_df = clean_df(pd.read_csv('05-recipe-summary.csv'))
orders_df = clean_df(pd.read_csv('06-orders-time-bucket.csv'))
git_df = clean_df(pd.read_csv('07-git-time-buckets.csv'))

# แปลงตัวเลข
inv_df['stock_qty'] = pd.to_numeric(inv_df['stock_qty'].astype(str).str.replace(',', ''), errors='coerce').fillna(0)
orders_df['order_qty'] = pd.to_numeric(orders_df['order_qty'], errors='coerce').fillna(0)
orders_df['bucket'] = pd.to_numeric(orders_df['bucket'], errors='coerce').fillna(0).astype(int)
git_df['git_qty'] = pd.to_numeric(git_df['git_qty'], errors='coerce').fillna(0)
git_df['bucket'] = pd.to_numeric(git_df['bucket'], errors='coerce').fillna(0).astype(int)
recipe_df['usage_percent'] = pd.to_numeric(recipe_df['usage_percent'], errors='coerce').fillna(0)
recipe_sum_df['total_cost'] = pd.to_numeric(recipe_sum_df['total_cost'], errors='coerce').fillna(0)

# ==========================================
# Session 2: Dictionaries & MILP
# ==========================================
rm_initial_inv = inv_df.set_index('rm_name')['stock_qty'].to_dict()
all_rms = sorted(list(rm_initial_inv.keys()))
git_grouped = git_df.groupby(['rm_name', 'bucket'])['git_qty'].sum().to_dict()
product_to_recipes = recipe_sum_df.groupby('product_name')['recipe_number'].apply(list).to_dict()
recipe_usage = recipe_df.set_index(['recipe_number', 'rm_name'])['usage_percent'].to_dict()
recipe_costs = recipe_sum_df.set_index('recipe_number')['total_cost'].to_dict()
all_buckets = sorted(list(set(orders_df['bucket'].unique()) | set(git_df['bucket'].unique())))

# Mapping MO Choice Variables
mo_choices = []
for idx, row in orders_df.iterrows():
    recs = product_to_recipes.get(row['product_name'], [])
    for r in recs: mo_choices.append((idx, r))
    mo_choices.append((idx, 'NONE'))

n_vars = len(mo_choices)
c = np.zeros(n_vars)
for i, (m_idx, r_name) in enumerate(mo_choices):
    if r_name == 'NONE': c[i] = 1e15 # Penalty
    else: c[i] = orders_df.loc[m_idx, 'order_qty'] * recipe_costs.get(r_name, 0)

A_rows = []; lb = []; ub = []
# 1 Choice per MO
for m_idx in orders_df.index:
    row = np.zeros(n_vars)
    for i, (v_idx, r_name) in enumerate(mo_choices):
        if v_idx == m_idx: row[i] = 1
    A_rows.append(row); lb.append(1); ub.append(1)

# Inventory Cumulative Constraint
for b_val in all_buckets:
    for rm_name in all_rms:
        row = np.zeros(n_vars)
        for i, (m_idx, r_name) in enumerate(mo_choices):
            if r_name != 'NONE' and orders_df.loc[m_idx, 'bucket'] <= b_val:
                row[i] = orders_df.loc[m_idx, 'order_qty'] * recipe_usage.get((r_name, rm_name), 0)
        supply_cum = rm_initial_inv.get(rm_name, 0) + sum(git_grouped.get((rm_name, bb), 0) for bb in all_buckets if bb <= b_val)
        A_rows.append(row); lb.append(-np.inf); ub.append(supply_cum)

res = milp(c=c, constraints=LinearConstraint(np.array(A_rows), lb, ub), integrality=np.ones(n_vars), bounds=Bounds(0, 1))

# ==========================================
# Session 3: Output Reports & Analysis
# ==========================================
selected = {m_idx: r_name for i, (m_idx, r_name) in enumerate(mo_choices) if res.x[i] > 0.5}

# --- 07-orders-time-bucket-calculation.csv ---
out_07 = orders_df.copy()
out_07['recipe_number'] = out_07.index.map(lambda i: selected[i] if selected[i] != 'NONE' else None)
out_07['is_producible'] = out_07.index.map(lambda i: True if selected[i] != 'NONE' else False)
out_07['order_cost'] = out_07['order_qty'] * out_07['recipe_number'].map(recipe_costs).fillna(0)
out_07.to_csv('07-orders-time-bucket-calculation.csv', index=False)

# --- 11-mo-shortage-details.csv (พร้อมบอกปริมาณ RM ที่ขาด) ---
shortage_details = []
for idx, row in out_07[out_07['is_producible'] == False].iterrows():
    mo_info = row.to_dict()
    # ใช้สูตรที่ถูกที่สุดเป็นตัวอ้างอิงเพื่อวิเคราะห์ว่าขาดอะไร
    recs = product_to_recipes.get(row['product_name'], [])
    r_best = sorted(recs, key=lambda r: recipe_costs.get(r, 0))[0] if recs else "N/A"
    mo_info['target_recipe'] = r_best

    if r_best != "N/A":
        for rm in all_rms:
            req = row['order_qty'] * recipe_usage.get((r_best, rm), 0)
            if req > 0:
                supply = rm_initial_inv.get(rm, 0) + sum(git_grouped.get((rm, bb), 0) for bb in all_buckets if bb <= row['bucket'])
                used = sum(orders_df.loc[m, 'order_qty'] * recipe_usage.get((selected[m], rm), 0)
                           for m in selected if selected[m] != 'NONE' and orders_df.loc[m, 'bucket'] <= row['bucket'])
                short_val = max(0, req - (supply - used))
                if short_val > 0: mo_info[f'shortage_{rm}'] = short_val
    shortage_details.append(mo_info)
pd.DataFrame(shortage_details).to_csv('11-mo-shortage-details.csv', index=False)

# --- 11-summary.csv ---
sum_metrics = [
     ["Total Produced Quantity", out_07[out_07['is_producible'] == True]['order_qty'].sum()],
    ["Total Order Cost (Produced)", out_07['order_cost'].sum()],
    ["Total Orders Count", len(out_07)],
    ["Producible Orders (Count)", len(out_07[out_07['is_producible'] == True])],
    ["Non-Producible Orders (Count)", len(out_07[out_07['is_producible'] == False])],
    ["Success Rate (%)", (len(out_07[out_07['is_producible'] == True]) / len(out_07)) * 100]
]
pd.DataFrame(sum_metrics, columns=["Metric", "Value"]).to_csv('11-summary.csv', index=False)

print("Status:", res.message)
print("Files generated: 07, 08, 09, 11-summary, 11-mo-shortage-details")

Status: Optimization terminated successfully. (HiGHS Status 7: Optimal)
Files generated: 07, 08, 09, 11-summary, 11-mo-shortage-details
